# 풀스택 GPT: #5.0 ~ #5.8
## Tasks:

- [x] 앞서 배운 메모리 클래스 중 하나를 사용하는 메모리로 LCEL 체인을 구현합니다.
- [x] 이 체인은 영화 제목을 가져와 영화를 나타내는 세 개의 이모티콘으로 응답해야 합니다. (예: "탑건" -> "🛩️👨‍✈️🔥". "대부" -> "👨‍👨‍👦🔫🍝").
- [x] 항상 세 개의 이모티콘으로 답장하도록 FewShotPromptTemplate 또는 FewShotChatMessagePromptTemplate을 사용하여 체인에 예시를 제공하세요.
    > - 요구조건에 맞는 답변 형식을 생성하도록 적절한 예시를 만들고, FewShotPromptTemplate 또는 FewShotChatMessagePromptTemplate를 이용하여 LLM에게 예시를 제공하세요.
    > - 자세한 사용법은 다음 공식 문서를 참고해보세요
        > - [Few-shot prompt templates](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples/)
        > - [Few-shot examples for chat models](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples_chat/)
- [x] 메모리가 작동하는지 확인하려면 체인에 두 개의 영화에 대해 질문한 다음 다른 셀에서 체인에 먼저 질문한 영화가 무엇인지 알려달라고 요청하세요.
    > - [x] ConversationBufferMemory 등 강의에서 배운 메모리 중 하나를 사용하여 이전 대화 기록을 기억하고 기록을 이용한 답변을 제공할 수 있도록 합니다.
    > - [x] 채팅 형식의 메모리 기록을 프롬프트에 추가하고 싶을 때는 MessagesPlaceholder를 이용하세요. (공식문서 예시)
    > - [x] RunnablePassthrough를 활용하면 LCEL 체인을 구현할 때 메모리 적용을 쉽게 할 수 있습니다. RunnablePassthrough는 메모리를 포함한 데이터를 체인의 각 단계에 전달하는 역할을 합니다. (강의 #5.7 1:04~ 참고)

In [ ]:
examples = [
    {
        "title": "Top Gun: Maverick",
        "answer": "🛩️👨‍✈️🔥"
    },
    {
        "title": "대부",
        "answer": "👨‍👨‍👦🔫🍝"
    },
    {
        "title": "Titanic", 
        "answer": "🚢💔🌊"
    },
    {
        "title": "Inception", 
        "answer": "🧠🌀💤"
    },
]

In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain.callbacks import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)
memory = ConversationBufferMemory(return_messages=True)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Take a movie {title} and represent it with three emojis."),
    ("ai", "{answer}"),
])

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie geek. You know plot of every movie so you can express it with emojis."),
    fewshot_prompt,
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(input):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | final_prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context({"input": question}, {"output":result.content})

invoke_chain("Wonka")
invoke_chain("Malena")

🍫🍭🎩👩‍🦰🇮🇹💔

In [3]:
invoke_chain("Which movie I asked right before?")

You asked about "Malena".